# **Introdução ao LangChain**

**Disciplina:** Generative AI & Advanced Analytics

**Instituição:** PUC Minas

**Professor:** Renan Santos Mendes

**Email:** renansantosmendes@gmail.com

---

Este notebook tem como objetivo apresentar de forma pratica os conceitos iniciais do LangChain, uma biblioteca para construcao de aplicacoes baseadas em modelos de linguagem (LLMs).

Serao abordados os seguintes topicos:

1. Configuracao do ambiente e do modelo de linguagem
2. Messages (SystemMessage, HumanMessage e AIMessage)
3. Templates (PromptTemplate e ChatPromptTemplate)
4. Runnables (conceitos basicos)
5. Chains (encadeamento de componentes)


## 1. Configuracao do ambiente

Antes de comecar, precisamos instalar a biblioteca `langchain-openai`, que fornece a integracao entre o LangChain e modelos compativeis com a API da OpenAI.

Nesta aula, usaremos um proxy proprio para acessar o modelo `gpt-4o-mini`, entao nao sera necessario fornecer uma chave de API valida.

A instalacao sera feita utilizando o `uv`, um gerenciador de pacotes mais rapido que o `pip` tradicional.

In [1]:
!uv pip install langchain-openai pgl-auth -q

## 2. Criando o modelo de linguagem

O `ChatOpenAI` e a classe do LangChain responsavel por representar um modelo de chat compativel com a API da OpenAI.

Abaixo, criamos uma instancia do modelo apontando para o proxy da disciplina.

In [2]:
from langchain_openai import ChatOpenAI
from google.colab import userdata
from pgl_auth import PGLAuthClient

token = PGLAuthClient().login(
    registration_number=userdata.get("PGL_REGISTRATION_NUMBER"),
    password=userdata.get("PGL_PASSWORD"),
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=token,
    temperature=0.2,
    base_url="https://pgl-proxy.vercel.app/v1",
)

In [3]:
from langchain_openai import ChatOpenAI
from google.colab import userdata
from pgl_auth import PGLAuthClient

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=token,
    temperature=0.2,
    base_url="https://pgl-proxy.vercel.app/v1",
)

Podemos testar o modelo enviando uma pergunta simples diretamente como texto.

In [ ]:
response = llm.invoke("me explique o que é uma função em python usando apenas uma frase")

In [ ]:
response

AIMessage(content='Uma função em Python é um bloco de código reutilizável que realiza uma tarefa específica e pode receber entradas (argumentos) e retornar uma saída (resultado).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 20, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c980d3075f', 'id': 'chatcmpl-EElDArQGTNaYgZ5ZWFUEHnZuQQmE4', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a01ca2-1665-7933-8df9-f80fe05186df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 32, 'total_tokens': 52, 'input_token_details': {'audio': 0, 'cache_read': 0},

In [ ]:
dialog = """
**Atendente (Mercado Livre):**
Olá, boa tarde! Meu nome é Juliana e vou acompanhar seu atendimento hoje. Em que posso ajudar?

**Cliente:**
Boa tarde. Estou extremamente insatisfeito. Comprei um notebook de aproximadamente R$ 8.500 e o sistema informa que foi entregue ontem às 15h42, mas eu não recebi absolutamente nada.

**Atendente:**
Sinto muito pelo ocorrido. Vou verificar todas as informações do seu pedido. Poderia confirmar o número do pedido, por favor?

**Cliente:**
Claro. É o pedido #MLB-845973221.
...

"""

In [ ]:
%%time
summary = llm.invoke(f"Resuma o diálogo a seguir extraindo os principais pontos do atendimento ao cliente: \n\n {dialog}")

In [ ]:
summary

In [ ]:
print(summary.content)

In [ ]:
print(response.content)

In [ ]:
prompt_1 = f"""
# Instruções
Resuma o diálogo a seguir extraindo os principais pontos do atendimento ao cliente.

O resumo deve ser sucinto e objtetivo. Traga somente o necessário e mais importante
do ponto de vista de negócio.

# Diálogo:

{dialog}
"""

prompt_2 = f"""
# Instruções
Entenda se a demanda/problema do cliente foi resolvida.

Sua resposta deve ser um json com apenas uma chave e um valor booleano no seguinte formato

'is_resolved': true_or_false

# Diálogo:

{dialog}
"""

In [ ]:
%%time
summary = llm.invoke(prompt_1)

In [ ]:
print(summary.content)

In [ ]:
problem_solved = llm.invoke(prompt_2)

In [ ]:
print(problem_solved.content)

## 3. Messages

Ao trabalhar com modelos de chat, a comunicacao e organizada em **mensagens**, cada uma com um papel especifico dentro da conversa. As tres principais mensagens do LangChain sao:

- **SystemMessage**: define o comportamento, o tom ou as regras que o modelo deve seguir durante toda a conversa. E como se fosse uma instrucao dada ao modelo antes do dialogo comecar.
- **HumanMessage**: representa a fala do usuario, ou seja, a pergunta ou solicitacao feita ao modelo.
- **AIMessage**: representa a resposta gerada pelo modelo. Tambem pode ser usada para simular respostas anteriores do modelo em um historico de conversa.

Vamos importar essas classes e construir uma conversa simples.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = [
    SystemMessage("Voce é um assistente que responde de forma breve e didatica."),
    HumanMessage("o que é uma variavel em programação")
]

response = llm.invoke(messages)
print(response.content)

Uma variável em programação é um espaço na memória do computador que armazena um valor. Esse valor pode ser alterado durante a execução do programa. As variáveis têm um nome, que é usado para referenciá-las, e podem armazenar diferentes tipos de dados, como números, texto ou booleanos (verdadeiro/falso). Por exemplo, em Python, você pode criar uma variável chamada `idade` e atribuir a ela um valor:

```python
idade = 25
```

Aqui, `idade` é a variável que armazena o valor `25`.


Podemos tambem simular um historico de conversa, incluindo uma resposta anterior do modelo (`AIMessage`) antes de fazer uma nova pergunta. Isso ajuda o modelo a manter contexto sobre o que ja foi dito.

In [ ]:
conversation_history = [
    SystemMessage("Voce é um assistente que responde de forma breve e didatica."),
    SystemMessage("Toda resposta que der, deve ser dada em japones"),
    HumanMessage("o que é uma variavel em programação"),
    AIMessage("Uma variável em programação é um espaço na memória do computador que armazena um valor"),
    HumanMessage("gere um exemplo em python")
    ]

response = llm.invoke(conversation_history)
print(response.content)

以下はPythonの変数の例です：

```python
# 変数の定義
名前 = "太郎"
年齢 = 25

# 変数の出力
print("名前:", 名前)
print("年齢:", 年齢)
```


## 4. Templates

Em aplicacoes reais, raramente escrevemos prompts fixos: normalmente queremos reaproveitar uma mesma estrutura de prompt, alterando apenas alguns valores. Para isso, o LangChain oferece os **templates**.

- **PromptTemplate**: usado para criar um template de texto simples, com variaveis que serao preenchidas dinamicamente.
- **ChatPromptTemplate**: usado para criar um template composto por varias mensagens (system, human, etc.), tambem com variaveis dinamicas.

Vamos comecar com o `PromptTemplate`.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("Explique o conceito de {concept} para um aluno iniciante em programação")

In [ ]:
prompt_template

PromptTemplate(input_variables=['concept'], input_types={}, partial_variables={}, template='Explique o conceito de {concept} para um aluno iniciante em programação')

In [ ]:
formatted_prompt = prompt_template.format(concept="loop")
print(formatted_prompt)

Explique o conceito de loop para um aluno iniciante em programação


In [ ]:
llm.invoke(formatted_prompt)

AIMessage(content='Claro! Vamos falar sobre o conceito de "loop" de uma forma simples.\n\nUm **loop** (ou laço) é uma estrutura de programação que permite que um conjunto de instruções seja repetido várias vezes, até que uma condição específica seja atendida. Em outras palavras, é uma maneira de fazer com que o computador execute a mesma tarefa várias vezes sem precisar escrever o mesmo código repetidamente.\n\nImagine que você está em uma festa e quer cumprimentar todos os seus amigos. Em vez de dizer "Oi" para cada um deles manualmente, você poderia usar um loop para fazer isso automaticamente. O loop continuaria a cumprimentar os amigos até que todos tivessem sido cumprimentados.\n\nExistem diferentes tipos de loops, mas os mais comuns são:\n\n1. **Loop "for"**: Esse tipo de loop é usado quando você sabe exatamente quantas vezes deseja repetir um bloco de código. Por exemplo, se você quiser contar de 1 a 5, você pode usar um loop "for" para fazer isso.\n\n   Exemplo em pseudocódigo:

Agora vamos usar o `ChatPromptTemplate`, que permite definir varias mensagens dentro do mesmo template, cada uma com seu papel (system, human, etc.).

In [4]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'Atue como um professor de computação que explica o conteúdo e mostra exemplos de código'),
    ('human', 'Explique o conceito de {concept} para um aluno iniciante em programação')
])

formatted_messages = chat_prompt_template.format_messages(concept="funcao")
for message in formatted_messages:
    print(message)

content='Atue como um professor de computação que explica o conteúdo e mostra exemplos de código' additional_kwargs={} response_metadata={}
content='Explique o conceito de funcao para um aluno iniciante em programação' additional_kwargs={} response_metadata={}


## 5. Runnables

Todo componente do LangChain que pode ser executado (um modelo, um template, uma funcao de transformacao de dados, entre outros) implementa a interface `Runnable`.

Um `Runnable` expoe metodos padronizados, como:

- `invoke`: executa o componente com uma unica entrada
- `batch`: executa o componente com varias entradas de uma vez
- `stream`: executa o componente retornando a resposta em partes (streaming)

Essa padronizacao e o que permite combinar diferentes componentes entre si, formando as **chains**, que veremos a seguir.

Abaixo, um exemplo simples mostrando que tanto o `ChatPromptTemplate` quanto o `llm` sao `Runnables`, pois ambos possuem o metodo `invoke`.

In [ ]:
print(hasattr(prompt_template, "invoke"))
print(hasattr(llm, "invoke"))

True
True


## 6. Chains

Uma **chain** e o encadeamento de dois ou mais `Runnables`, de forma que a saida de um componente seja usada como entrada do proximo.

No LangChain, esse encadeamento e feito de maneira declarativa usando o operador `|` (pipe), conhecido como LCEL (LangChain Expression Language).

Abaixo, vamos criar uma chain simples que:

1. Recebe um conceito
2. Formata o prompt usando o `ChatPromptTemplate`
3. Envia o prompt formatado para o modelo `llm`

In [5]:
chain = chat_prompt_template | llm

In [6]:
chain

ChatPromptTemplate(input_variables=['concept'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Atue como um professor de computação que explica o conteúdo e mostra exemplos de código'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['concept'], input_types={}, partial_variables={}, template='Explique o conceito de {concept} para um aluno iniciante em programação'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, '

In [7]:
type(chain)

langchain_core.runnables.base.RunnableSequence

In [ ]:
response = chain.invoke({"concept": "recursao"})
print(response.content)

Claro! Vamos falar sobre recursão de uma maneira simples.

### O que é Recursão?

Recursão é uma técnica de programação onde uma função chama a si mesma para resolver um problema. Essa abordagem é útil para resolver problemas que podem ser divididos em subproblemas menores e semelhantes.

### Como Funciona?

Quando uma função recursiva é chamada, ela executa seu código e, em algum ponto, pode chamar a si mesma com um novo argumento. Para evitar que a função entre em um loop infinito, é importante ter uma **condição de parada**. Essa condição diz à função quando parar de se chamar.

### Exemplo Clássico: Fatorial

Um exemplo clássico de recursão é o cálculo do fatorial de um número. O fatorial de um número \( n \) (denotado como \( n! \)) é o produto de todos os números inteiros de 1 até \( n \).

A definição recursiva do fatorial é:
- \( n! = n \times (n-1)! \) para \( n > 1 \)
- \( 1! = 1 \)
- \( 0! = 1 \)

### Implementação em Código

Aqui está um exemplo de como implementar a função

Podemos tambem adicionar mais um passo a chain, por exemplo, extraindo apenas o texto da resposta usando o `StrOutputParser`, que converte a saida do modelo (um `AIMessage`) em uma string simples.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain_with_parser = chat_prompt_template | llm |

result = chain_with_parser.invoke({"concept": "lista encadeada"})
print(result)
print(type(result))

## 7. Runnables avançados: RunnableParallel e RunnableBranch

Alem do encadeamento simples com o operador `|`, o LangChain oferece componentes que permitem organizar a execucao dos `Runnables` de formas mais elaboradas. Dois exemplos bastante uteis sao:

- **RunnableParallel**: executa varios `Runnables` ao mesmo tempo, usando a mesma entrada, e retorna um dicionario com o resultado de cada um. E util quando queremos, por exemplo, gerar respostas diferentes para o mesmo conceito (uma explicacao e um exemplo de codigo, ao mesmo tempo).
- **RunnableBranch**: permite definir diferentes caminhos de execucao (chains diferentes) de acordo com uma condicao aplicada sobre a entrada. Funciona como uma estrutura de `if / elif / else` para `Runnables`.

Vamos ver um exemplo simples de cada um.

### 7.1 RunnableParallel

No exemplo abaixo, vamos executar duas chains diferentes ao mesmo tempo, a partir do mesmo conceito de entrada:

- uma chain que gera uma explicacao teorica
- uma chain que gera um exemplo de codigo em Python

O resultado sera um dicionario contendo as duas respostas.

In [ ]:
from langchain_core.runnables import RunnableParallel

explanation_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de programacao."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

code_example_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de programacao."),
    ("human", "Escreva um exemplo curto de codigo em Python sobre {concept}."),
])

explanation_chain = explanation_prompt | llm | StrOutputParser()
code_example_chain = code_example_prompt | llm | StrOutputParser()

parallel_chain = ...

parallel_result = parallel_chain.invoke({"concept": "list comprehension"})
print(parallel_result["explanation"])
print("---")
print(parallel_result["code_example"])

### 7.2 RunnableBranch

No exemplo abaixo, vamos criar duas chains diferentes: uma especializada em conceitos de programacao e outra especializada em conceitos de matematica. O `RunnableBranch` sera responsavel por escolher qual chain executar, de acordo com o valor do campo `topic` presente na entrada.

Cada condicao do `RunnableBranch` e uma tupla no formato `(funcao_condicao, chain)`. A ultima entrada, sem condicao, funciona como o caminho padrao (`else`).

In [ ]:
from langchain_core.runnables import RunnableBranch

programming_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de programacao."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

math_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um professor de matematica."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

default_prompt = ChatPromptTemplate.from_messages([
    ("system", "Voce e um assistente generalista."),
    ("human", "Explique de forma breve o conceito de {concept}."),
])

programming_chain = programming_prompt | llm | StrOutputParser()
math_chain = math_prompt | llm | StrOutputParser()
default_chain = default_prompt | llm | StrOutputParser()

branch_chain =...

programming_result = branch_chain.invoke({"topic": "programming", "concept": "recursao"})
print(programming_result)
print("---")
math_result = branch_chain.invoke({"topic": "math", "concept": "derivada"})
print(math_result)